In [0]:
#Source
def selective_df_1(configuration,query=None,table=None,IsDeltaTable=False):
    print("Query: ",query)
    print("Table: ",table)
    if query != "" and query is not None:
        createdf =spark.read.format("jdbc")\
        .option("url",f"jdbc:sqlserver://{configuration['server']}:{configuration['port']};databaseName={configuration['database']};trustServerCertificate=true")\
        .option("query",query)\
        .option("user",configuration['username'])\
        .option("password",configuration['password'])\
        .option("driver",configuration['driver'])\
        .load()
    elif (table != "" and table is not None):
        createdf =spark.read.format("jdbc")\
        .option("url",f"jdbc:sqlserver://{configuration['server']}:{configuration['port']};databaseName={configuration['database']};trustServerCertificate=true")\
        .option("dbtable",table)\
        .option("user",configuration['username'])\
        .option("password",configuration['password'])\
        .option("driver",configuration['driver'])\
        .load()
    elif (IsDeltaTable==True):
        createdf=ExecuteDeltaQuery(configuration, query)
    else:
        raise ValueError("Either 'query' or 'table' must be provided")
    return createdf

In [0]:
#Source
def selective_df(configuration,query=None,table=None,IsDeltaTable=False):
    print("Query: ",query)
    print("Table: ",table)
    if query != "" and query is not None:
        createdf =spark.read.format("jdbc")\
        .option("url",f"jdbc:sqlserver://{configuration['server']}:{configuration['port']};databaseName={configuration['database']};authentication=ActiveDirectoryServicePrincipal;encrypt=true;trustServerCertificate=true")\
        .option("query",query)\
        .option("user",configuration['username'])\
        .option("password",configuration['password'])\
        .option("driver",configuration['driver'])\
        .load()
    elif (table != "" and table is not None):
        createdf =spark.read.format("jdbc")\
        .option("url",f"jdbc:sqlserver://{configuration['server']}:{configuration['port']};databaseName={configuration['database']};authentication=ActiveDirectoryServicePrincipal;encrypt=true;trustServerCertificate=true")\
        .option("dbtable",table)\
        .option("user",configuration['username'])\
        .option("password",configuration['password'])\
        .option("driver",configuration['driver'])\
        .load()
    elif (IsDeltaTable==True):
        createdf=ExecuteDeltaQuery(configuration, query)
    else:
        raise ValueError("Either 'query' or 'table' must be provided")
    return createdf
 

In [0]:
def ExecuteDeltaQuery(configuration, sql_command1,sql_command2 =""):
   
    # Set the current catalog and database context
    catelogsql=f"USE CATALOG '{configuration['server']}'"
    databasesql=f"USE DATABASE '{configuration['database']}"
    spark.sql(catelogsql)
    spark.sql(databasesql)
    
    try:
        # Check if the SQL command is a SELECT query
        if sql_command1.strip().upper().startswith("SELECT"):
            #For SELECT queries, return the DataFrame
            return spark.sql(sql_command1 + " " + sql_command2)
        else:
        #For DDL/DML commands, run the command and return a success message
            spark.sql(sql_command1 + " " + sql_command2)
            return "SQL command executed successfully."
    except Exception as e:
        return f"Error executing SQL command: {str(e)}"

In [0]:
def save_df_1(paramdf, configuration, table=None):	
    paramdf.write.format("jdbc")\
    .mode("append")\
    .option("url", f"jdbc:sqlserver://{configuration['server']}:{configuration['port']};databaseName={configuration['database']};trustServerCertificate=true")\
    .option("dbtable", table)\
    .option("user", configuration['username'])\
    .option("password", configuration['password'])\
    .option("driver", configuration['driver'])\
    .option("batchsize", "1000")\
    .option("numPartitions", 4)\
    .option("truncate", "false")\
    .option("encoding", "windows-1252")\
    .save()

In [0]:
def save_df(paramdf, configuration, table=None):	
    paramdf.write.format("jdbc")\
    .mode("append")\
    .option("url",f"jdbc:sqlserver://{configuration['server']}:{configuration['port']};databaseName={configuration['database']};authentication=ActiveDirectoryServicePrincipal;encrypt=true;trustServerCertificate=true")\
    .option("dbtable", table)\
    .option("user", configuration['username'])\
    .option("password", configuration['password'])\
    .option("driver", configuration['driver'])\
    .option("batchsize", "1000")\
    .option("numPartitions", 4)\
    .option("truncate", "false")\
    .option("encoding", "windows-1252")\
    .save()

In [0]:

def execute_pyodbc(configuration,query):
    try:
        import pyodbc
        conn = pyodbc.connect(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={configuration['server']},{configuration['port']};"
        f"DATABASE={configuration['database']};"
        f"UID={configuration['username']};"
        f"PWD={configuration['password']};"
        f"Authentication=ActiveDirectoryServicePrincipal;"
        f"Encrypt=yes;"
        f"TrustServerCertificate=yes;"
        )
 
    # Create a cursor object
        cursor = conn.cursor()
 
# Execute a query
        data1=cursor.execute(query)
        while cursor.nextset():
            pass     
        conn.commit()
        #data=data1.fetchall()
    except Exception as e:
        return(f"Error executing SQL command: {str(e)}")        
    finally:
        cursor.close()
        conn.close()
    

In [0]:
# def execute_pyodbc(configuration,query):
#     try:
#         import pyodbc
#         conn = pyodbc.connect(f"DRIVER={{ODBC Driver 17 for SQL Server}}; Server={configuration['server']},{configuration['port']}; DATABASE={configuration['database']};UID={configuration['username']};PWD={configuration['password']};")
 
#     # Create a cursor object
#         cursor = conn.cursor()
 
# # Execute a query
#         data1=cursor.execute(query)
#         while cursor.nextset():
#             pass     
#         conn.commit()
#         #data=data1.fetchall()
#     except Exception as e:
#         return(f"Error executing SQL command: {str(e)}")        
#     finally:
#         cursor.close()
#         conn.close()

In [0]:
def write_df_sql_mi(configuration,table=None,df=None):
    
    print("Table: ",table)
    if df is not None and table is not None:
        df.write.format("jdbc").option("url",f"jdbc:sqlserver://{configuration['server']}:{configuration['port']};databaseName={configuration['database']};trustServerCertificate=true").option("dbtable",table).option("user",configuration['username']).option("password",configuration['password']).option("driver",configuration['driver']).option("batchsize","10000").mode("append").save()
        # df.show()
        return table

def pyodbc_connection(configuration,query):
    import pyodbc
    conn = pyodbc.connect(f"DRIVER={{ODBC Driver 17 for SQL Server}}; Server={configuration['server']},{configuration['port']}; DATABASE={configuration['database']};UID={configuration['username']};PWD={configuration['password']};")
 
    # Create a cursor object
    cursor = conn.cursor()
 
# Execute a query
    data1=cursor.execute(query)
    data=data1.fetchall()
    column=[column[0] for column in cursor.description]
    # df2=spark.createDataFrame(data, column)
    cursor.close()
    conn.close()
    return data,column

        

In [0]:
def pyodbc_Execute_temp(configuration,query,query1):
    try:
        import pyodbc
        conn = pyodbc.connect(f"DRIVER={{ODBC Driver 17 for SQL Server}}; Server={configuration['server']},{configuration['port']}; DATABASE={configuration['database']};UID={configuration['username']};PWD={configuration['password']};",autocommit=True,timeout=100000)
 
    # Create a cursor object
        cursor = conn.cursor()
 
# Execute a query
        
        data1=cursor.execute(query)
        while cursor.nextset():
            pass     
        conn.commit()
        
        data1=cursor.execute(query1)
        data=data1.fetchall()
        column=[column[0] for column in cursor.description]
        #data=data1.fetchall()
        return data
    except Exception as e:
        return(f"Error executing SQL command: {str(e)}")        
    finally:
        cursor.close()
        conn.close()

In [0]:
# def execute_pyodbc(configuration,query):
#     try:
#         import pyodbc
#         conn = pyodbc.connect(f"DRIVER={{ODBC Driver 17 for SQL Server}}; Server={configuration['server']},{configuration['port']}; DATABASE={configuration['database']};UID={configuration['username']};PWD={configuration['password']};")
 
#     # Create a cursor object
#         cursor = conn.cursor()
 
# # Execute a query
#         data1=cursor.execute(query)
#         while cursor.nextset():
#             pass     
#         conn.commit()
#         #data=data1.fetchall()
#     except Exception as e:
#         return(f"Error executing SQL command: {str(e)}")        
#     finally:
#         cursor.close()
#         conn.close()

In [0]:
def pyodbc_Execute(configuration,query,query1):
    try:
        import pyodbc
        conn = pyodbc.connect(f"DRIVER={{ODBC Driver 17 for SQL Server}}; Server={configuration['server']},{configuration['port']}; DATABASE={configuration['database']};UID={configuration['username']};PWD={configuration['password']};")
 
    # Create a cursor object
        cursor = conn.cursor()
 
# Execute a query
        data1=cursor.execute(query)
        while cursor.nextset():
            pass     
        conn.commit()
        data1=cursor.execute(query1)
        data=data1.fetchall()
        column=[column[0] for column in cursor.description]
        #data=data1.fetchall()
        return data
    except Exception as e:
        return(f"Error executing SQL command: {str(e)}")        
    finally:
        cursor.close()
        conn.close()
        